# 3D PixelShuffle (Sub-Pixel Convolution) Triton GPU Kernel Benchmark

This notebook demonstrates a custom, high-performance **Triton** GPU kernel for **3D PixelShuffle** (sub-pixel convolution), isolated and benchmarked against PyTorch's native implementation.

### Why Custom Triton?
In 3D super-resolution, sub-pixel convolution is normally implemented in PyTorch using `x.view(...).permute(...).contiguous()`. 
1. **Memory Overhead**: The `permute` operation changes the strides, making the tensor non-contiguous. Calling `.contiguous()` forces a new memory allocation and a full copy of the data.
2. **Triton Speedup**: By writing a custom Triton kernel, we can calculate the destination memory address in parallel for each output thread, loading from the input and storing directly to the output in a single fused GPU operation, bypassing the allocation and copy overhead of intermediate views.

## 0. Install dependencies

Ensure you are running on a **GPU runtime** in Colab (Runtime -> Change runtime type -> GPU).

In [ ]:
# Install Triton if not already installed (useful in Google Colab)
import sys
import torch

if 'google.colab' in sys.modules:
    !pip install -q triton

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Implementations

We define the PyTorch baseline and our custom Triton GPU kernel wrapper.

In [ ]:
import torch
from torch import nn
import triton
import triton.language as tl

# --- PyTorch Native 3D PixelShuffle ---
class PyTorchPixelShuffle3d(nn.Module):
    def __init__(self, upscale_factor: int):
        super().__init__()
        self.r = upscale_factor

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, d, h, w = x.shape
        r = self.r
        oc = c // (r ** 3)
        x = x.view(b, oc, r, r, r, d, h, w)
        x = x.permute(0, 1, 5, 2, 6, 3, 7, 4).contiguous()
        return x.view(b, oc, d * r, h * r, w * r)


# --- Triton Custom 3D PixelShuffle Kernel ---
@triton.jit
def _pixelshuffle3d_kernel(
    x_ptr,
    y_ptr,
    stride_xb, stride_xc, stride_xd, stride_xh, stride_xw,
    stride_yb, stride_yc, stride_yd, stride_yh, stride_yw,
    B, OC, D, H, W,
    r,
    NUM_ELEMENTS,
    BLOCK_SIZE: tl.constexpr
):
    pid = tl.program_id(0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < NUM_ELEMENTS
    
    # Output spatial dims
    OD = D * r
    OH = H * r
    OW = W * r
    
    # Decompose 1D output offset into 5D coordinates (b, oc, od, oh, ow)
    ow = offsets % OW
    temp = offsets // OW
    oh = temp % OH
    temp = temp // OH
    od = temp % OD
    temp = temp // OD
    oc = temp % OC
    b = temp // OC
    
    # Compute input coordinates (id, ih, iw) and sub-pixel offsets (rd, rh, rw)
    id = od // r
    rd = od % r
    
    ih = oh // r
    rh = oh % r
    
    iw = ow // r
    rw = ow % r
    
    # Compute input channel index ic corresponding to (oc, rd, rh, rw)
    # Match PyTorch permute(0, 1, 5, 2, 6, 3, 7, 4) mapping
    ic = oc * (r * r * r) + rd * (r * r) + rh * r + rw
    
    # Compute 1D pointer offsets
    x_offset = b * stride_xb + ic * stride_xc + id * stride_xd + ih * stride_xh + iw * stride_xw
    y_offset = b * stride_yb + oc * stride_yc + od * stride_yd + oh * stride_yh + ow * stride_yw
    
    # Memory transfer
    x_vals = tl.load(x_ptr + x_offset, mask=mask)
    tl.store(y_ptr + y_offset, x_vals, mask=mask)


def triton_pixel_shuffle_3d(x: torch.Tensor, upscale_factor: int) -> torch.Tensor:
    assert x.is_cuda, "Input tensor must be on CUDA to use Triton!"
    b, c, d, h, w = x.shape
    r = upscale_factor
    
    assert c % (r ** 3) == 0, f"Channel count {c} is not divisible by upscale_factor^3 ({r**3})!"
    oc = c // (r ** 3)
    
    od, oh, ow = d * r, h * r, w * r
    y = torch.empty((b, oc, od, oh, ow), device=x.device, dtype=x.dtype)
    
    num_elements = y.numel()
    
    # Define grid size
    grid = lambda meta: (triton.cdiv(num_elements, meta['BLOCK_SIZE']),)
    
    _pixelshuffle3d_kernel[grid](
        x, y,
        x.stride(0), x.stride(1), x.stride(2), x.stride(3), x.stride(4),
        y.stride(0), y.stride(1), y.stride(2), y.stride(3), y.stride(4),
        b, oc, d, h, w,
        r,
        num_elements,
        BLOCK_SIZE=1024
    )
    return y

## 2. Correctness Validation

We verify that the Triton custom kernel outputs match PyTorch's native `PixelShuffle3d` outputs exactly.

In [ ]:
if torch.cuda.is_available():
    # Test dimensions
    B, OC, r = 2, 8, 2
    C = OC * (r**3) # 64 channels
    D, H, W = 16, 16, 16
    
    # Create random test input
    x = torch.randn((B, C, D, H, W), device='cuda', dtype=torch.float32)
    
    # Run PyTorch Native
    pt_shuffle = PyTorchPixelShuffle3d(upscale_factor=r)
    y_pytorch = pt_shuffle(x)
    
    # Run Triton Kernel
    y_triton = triton_pixel_shuffle_3d(x, upscale_factor=r)
    
    # Validate
    correct = torch.allclose(y_pytorch, y_triton, atol=1e-6)
    max_diff = (y_pytorch - y_triton).abs().max().item()
    
    print(f"Validation Results:")
    print(f"  Outputs match exactly: {correct}")
    print(f"  Max absolute difference: {max_diff:.3e}")
    assert correct, "Triton output does not match PyTorch output!"
else:
    print("Skipping validation: No GPU available.")

## 3. Latency Benchmarking

We evaluate performance across common 3D Patch dimensions (`16^3`, `32^3`, `64^3`) using `triton.testing.do_bench`.

In [ ]:
import pandas as pd

if torch.cuda.is_available():
    results = []
    r = 2
    OC = 8
    C = OC * (r**3) # 64 channels
    
    sizes = [16, 32, 64]
    pt_shuffle = PyTorchPixelShuffle3d(upscale_factor=r)
    
    print("Benchmarking latencies (ms)...\n")
    for size in sizes:
        x = torch.randn((2, C, size, size, size), device='cuda', dtype=torch.float32)
        
        # Benchmark PyTorch
        ms_pytorch = triton.testing.do_bench(lambda: pt_shuffle(x))
        
        # Benchmark Triton
        ms_triton = triton.testing.do_bench(lambda: triton_pixel_shuffle_3d(x, upscale_factor=r))
        
        speedup = ms_pytorch / ms_triton
        
        results.append({
            "Grid Size": f"{size}x{size}x{size}",
            "PyTorch (ms)": f"{ms_pytorch:.4f}",
            "Triton (ms)": f"{ms_triton:.4f}",
            "Speedup Factor": f"{speedup:.2fx}"
        })
        
    df = pd.DataFrame(results)
    print(df.to_markdown(index=False))
else:
    print("Skipping benchmarks: No GPU available.")

## 4. Visualization of Speedup

In [ ]:
import matplotlib.pyplot as plt

if torch.cuda.is_available() and len(results) > 0:
    sizes_str = [r["Grid Size"] for r in results]
    pt_times = [float(r["PyTorch (ms)"]) for r in results]
    tr_times = [float(r["Triton (ms)"]) for r in results]
    
    x_indices = range(len(sizes_str))
    width = 0.35
    
    plt.figure(figsize=(10, 5), dpi=100)
    ax = plt.gca()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Draw bars
    plt.bar([idx - width/2 for idx in x_indices], pt_times, width, label='PyTorch Native', color='#2B5C8F')
    plt.bar([idx + width/2 for idx in x_indices], tr_times, width, label='Triton Kernel', color='#D95D39')
    
    plt.title('3D PixelShuffle Speed Benchmark: PyTorch vs. Triton Custom Kernel', fontsize=13, fontweight='semibold', pad=15)
    plt.xlabel('3D Input Patch Size', fontsize=11, labelpad=8)
    plt.ylabel('Execution Latency (ms)', fontsize=11, labelpad=8)
    plt.xticks(x_indices, sizes_str)
    
    plt.grid(True, axis='y', linestyle='--', alpha=0.5, color='#BDC3C7')
    plt.legend(frameon=True, facecolor='white', edgecolor='none', fontsize=11)
    plt.tick_params(axis='both', which='major', labelsize=10)
    
    # Add speedup annotations
    for idx, (pt, tr) in enumerate(zip(pt_times, tr_times)):
        factor = pt / tr
        plt.text(idx, max(pt, tr) * 1.05, f"{factor:.1f}x Speedup", ha='center', fontsize=10, fontweight='bold', color='#2C3E50')
        
    plt.ylim(0, max(pt_times) * 1.2)
    plt.tight_layout()
    plt.show()
else:
    print("Skipping plot: No benchmark data available.")

## 5. Visualizing GPU Memory & Compiled Triton IR (TTIR / TTGIR)

To analyze hardware utilization (HBM to SRAM memory transfer, register mapping, and shared memory usage), Triton compiles your kernel into several intermediate representations (IR):

1. **TTIR (Triton IR)**: High-level compiler representation, showing abstract grid-level loops.
2. **TTGIR (Triton GPU IR)**: GPU-specific representation mapping threads to registers and allocating **SRAM (Shared Memory)**.

We can inspect these directly from the Python objects on any CUDA GPU runtime!

In [ ]:
if torch.cuda.is_available():
    print("=== Triton GPU Intermediate Representation (TTGIR) ===")
    # Print the compiled assembly / IR which contains the register assignment and SRAM shmem sizing!
    print(_pixelshuffle3d_kernel.asm["ttgir"])
else:
    print("No GPU available to run Triton JIT compilation.")

### Profiling Memory Hierarchy Transfers (HBM & SRAM) with Triton-Viz

For interactive visual debugging of tiling, thread layouts, and memory loads/stores between HBM (Global Memory) and SRAM (Shared Memory), you can use the open-source **Triton-Viz** tool:

```bash
# Install Triton-Viz (requires WSL on Windows, native on Linux)
pip install triton-viz
```

To generate a complete memory access trace visualization, you can import `triton_viz` and wrap your kernel execution as follows:

```python
import triton_viz

# 1. Enable tracing
triton_viz.trace(triton_pixel_shuffle_3d)

# 2. Launch the kernel on a test input
x = torch.randn((2, 64, 16, 16, 16), device='cuda', dtype=torch.float32)
y = triton_pixel_shuffle_3d(x, upscale_factor=2)

# 3. Export to a visual HTML file
triton_viz.launch()
```
This trace maps every load from HBM, layout in SRAM, and store back to HBM, displaying an interactive grid of thread execution blocks!